# Football Detection + Tracking + Otomatik Takım Ayrımı

Bu notebook Roboflow modelinden gelen `ball / player / goalkeeper / referee` tespitlerini kullanır, ByteTrack ile takip eder ve **player** nesnelerini forma rengine göre otomatik olarak `team_1` / `team_2` diye ayırır.

Eklenenler:
- Oyuncu crop'ının üst gövde/forma bölgesinden renk özelliği çıkarılır.
- İlk yeterli örnekten sonra 2 kümeli KMeans ile takım renk modeli kurulur.
- Her `track_id` için çoğunluk oylaması ile takım etiketi sabitlenir.
- Oyuncu elipsleri takım rengine göre farklı çizilir.
- CSV çıktısına `team_id`, `team_label`, `foot_x`, `foot_y` kolonları eklenir.

Not: En iyi sonuç için `DETECT_EVERY_N_FRAMES = 1` önerilir. Bu daha yavaş çalışır ama oyuncu/takım istatistiği için daha sağlıklıdır.


In [ ]:
!pip -q install inference-sdk supervision opencv-python-headless pandas tqdm


## 1) Ayarlar

İlk test için:
- `DETECT_EVERY_N_FRAMES = 5` hızlıdır ama kutular/ID'ler birkaç frame gecikmeli görünebilir.
- En akıcı ve doğru takip için `DETECT_EVERY_N_FRAMES = 1` yap. Bu daha yavaş ve daha fazla API kullanır.
- Çıktı FPS'i kaynak videodan otomatik alınır.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import cv2
import json
import math
import time
import pandas as pd
import numpy as np
from collections import defaultdict, deque, Counter
from tqdm.notebook import tqdm
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode

from inference_sdk import InferenceHTTPClient

# ======================
# MODEL AYARLARI
# ======================
MODEL_ID = "football-players-detection-3zvbc/11"
API_URL = "https://serverless.roboflow.com"

# Roboflow API key'i buraya direkt yazma. Çalıştırınca gizli olarak gir.
ROBOFLOW_API_KEY = getpass("Roboflow API Key: ")

CLIENT = InferenceHTTPClient(
    api_url=API_URL,
    api_key=ROBOFLOW_API_KEY
)

# ======================
# VIDEO / PERFORMANS AYARLARI
# ======================
# İstatistik ve takım ayrımı için 1 önerilir. Hızlı test için 3 veya 5 yapabilirsin.
DETECT_EVERY_N_FRAMES = 1
MAX_SECONDS = 30               # None yaparsan tüm videoyu işler.
CONFIDENCE_THRESHOLD = 0.20     # Oyuncular kaçıyorsa 0.15-0.20 deneyebilirsin.

# ======================
# TAKIM AYRIMI AYARLARI
# ======================
TEAM_CLASSIFICATION_ENABLED = True
CLASSIFY_GOALKEEPERS = False   # Kaleciler farklı forma giydiği için ilk aşamada False önerilir.
TEAM_MIN_SAMPLES = 24          # Model kurulmadan önce toplanacak minimum player renk örneği.
TEAM_MAX_SAMPLES = 500         # KMeans için bellekte tutulacak maksimum örnek.
TEAM_REFIT_EVERY = 25          # Kaç yeni örnekte bir takım merkezleri güncellensin.
TEAM_MIN_TRACK_VOTES = 3       # Track'e takım etiketi vermeden önce gereken minimum oy.

TEAM_1_NAME = "team_1"
TEAM_2_NAME = "team_2"
UNKNOWN_TEAM_NAME = "unknown"

# OpenCV BGR renkleri. İstersen buradan değiştir.
TEAM_1_DRAW_COLOR = (255, 0, 0)    # mavi
TEAM_2_DRAW_COLOR = (0, 0, 255)    # kırmızı
UNKNOWN_PLAYER_COLOR = (255, 255, 0)  # cyan
BALL_COLOR = (0, 255, 0)
REFEREE_COLOR = (0, 220, 255)
GOALKEEPER_COLOR = (255, 100, 255)

# Görsel çizim ayarları
SHOW_CLASS_NAME = False
SHOW_CONFIDENCE = False
SHOW_TRACK_ID = True
SHOW_TEAM_LABEL = False        # True yaparsan ID yanında T1/T2 de yazar.
DRAW_BOXES = False
DRAW_ELLIPSES = True
DRAW_BALL_TRIANGLE = True


# ======================
# STABIL ID / ID SWITCH AZALTMA AYARLARI
# ======================
# ByteTrack bazen oyuncu çakışmalarında raw ID değiştirir. Bu katman raw ID'leri daha stabil "stable_id"ye çevirir.
STABLE_ID_ENABLED = True
SHOW_STABLE_ID = True          # True: videoda stable_id gösterilir. False: ByteTrack raw ID gösterilir.
STABLE_MAX_GAP_SECONDS = 1.20  # Oyuncu kısa süre kaybolursa kaç saniye içinde aynı kişi kabul edilsin.
STABLE_PLAYER_MATCH_DISTANCE_PX = 135  # 1080p geniş açı için makul. ID bölünüyorsa artır, yanlış birleşiyorsa azalt.
STABLE_BALL_MATCH_DISTANCE_PX = 90
STABLE_REFEREE_MATCH_DISTANCE_PX = 120
STABLE_TEAM_STRICT = True      # Farklı takım etiketlerini birleştirmez; karışmayı azaltır.

# Çıktı klasörü
OUTPUT_DIR = Path("/content/football_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2) Video yükle

In [ ]:
uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("Video yüklenmedi.")

VIDEO_PATH = list(uploaded.keys())[0]
print("Yüklenen video:", VIDEO_PATH)

## 3) Roboflow sonucunu Detection formatına çeviren yardımcı fonksiyonlar

In [ ]:
def roboflow_predictions_to_xyxy(result, confidence_threshold=0.25):
    """
    Roboflow inference result -> dict list
    """
    preds = result.get("predictions", [])
    detections = []

    for p in preds:
        conf = float(p.get("confidence", 0))
        if conf < confidence_threshold:
            continue

        x = float(p["x"])
        y = float(p["y"])
        w = float(p["width"])
        h = float(p["height"])

        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2

        detections.append({
            "xyxy": [x1, y1, x2, y2],
            "confidence": conf,
            "class_name": str(p.get("class", "object"))
        })

    return detections


def class_to_id(class_name):
    mapping = {
        "ball": 0,
        "player": 1,
        "goalkeeper": 2,
        "referee": 3
    }
    return mapping.get(class_name, 99)


def detections_to_supervision(detections):
    import supervision as sv

    if len(detections) == 0:
        return sv.Detections.empty()

    xyxy = np.array([d["xyxy"] for d in detections], dtype=np.float32)
    confidence = np.array([d["confidence"] for d in detections], dtype=np.float32)
    class_id = np.array([class_to_id(d["class_name"]) for d in detections], dtype=int)

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        class_id=class_id
    )


def get_class_name_from_id(class_id):
    reverse = {
        0: "ball",
        1: "player",
        2: "goalkeeper",
        3: "referee"
    }
    return reverse.get(int(class_id), "object")

## 4) Forma rengine göre takım ayrımı ve çizim fonksiyonları

Takım ayrımı model class'ını değiştirmez. Model hâlâ `player` tespit eder; bu hücre `player` crop'ından forma rengini okuyup `team_1` / `team_2` etiketi üretir.


In [ ]:
def clip_xyxy(xyxy, frame_shape):
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = map(float, xyxy)
    x1 = int(max(0, min(w - 1, x1)))
    y1 = int(max(0, min(h - 1, y1)))
    x2 = int(max(0, min(w - 1, x2)))
    y2 = int(max(0, min(h - 1, y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def extract_jersey_feature(frame, xyxy):
    """
    Oyuncunun üst gövde/forma bölgesinden renk özelliği çıkarır.
    Dönüş: LAB renk uzayında 3 boyutlu feature veya None.
    """
    clipped = clip_xyxy(xyxy, frame.shape)
    if clipped is None:
        return None

    x1, y1, x2, y2 = clipped
    bw, bh = x2 - x1, y2 - y1
    if bw < 6 or bh < 12:
        return None

    # Forma bölgesi: kutunun üst-orta kısmı. Saha, şort ve ayakkabı etkisini azaltır.
    rx1 = x1 + int(bw * 0.20)
    rx2 = x1 + int(bw * 0.80)
    ry1 = y1 + int(bh * 0.15)
    ry2 = y1 + int(bh * 0.58)
    crop = frame[ry1:ry2, rx1:rx2]
    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    # Yeşil saha piksellerini ve çok karanlık pikselleri azalt.
    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    grass_mask = (h >= 35) & (h <= 85) & (s >= 35) & (v >= 35)
    valid_mask = (~grass_mask) & (v >= 25)

    pixels = crop[valid_mask]
    if len(pixels) < 10:
        pixels = crop.reshape(-1, 3)
    if len(pixels) == 0:
        return None

    # Gürültüye dayanıklı olması için median renk kullan.
    median_bgr = np.median(pixels, axis=0).astype(np.uint8).reshape(1, 1, 3)
    lab = cv2.cvtColor(median_bgr, cv2.COLOR_BGR2LAB).reshape(3).astype(np.float32)
    return lab


class OnlineTeamClassifier:
    """
    Player crop renklerinden 2 takım modeli kurar.
    Track bazında majority voting ile takım etiketini stabil tutar.
    """
    def __init__(self, min_samples=24, max_samples=500, refit_every=25, min_track_votes=3):
        self.min_samples = min_samples
        self.max_samples = max_samples
        self.refit_every = refit_every
        self.min_track_votes = min_track_votes
        self.features = deque(maxlen=max_samples)
        self.centers = None
        self.new_samples_since_fit = 0
        self.track_votes = defaultdict(Counter)
        self.track_id_to_team = {}

    @property
    def ready(self):
        return self.centers is not None

    def _fit(self):
        if len(self.features) < self.min_samples:
            return

        data = np.array(self.features, dtype=np.float32)
        if len(data) < 2:
            return

        # cv2.kmeans: 2 takım kümesi
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.2)
        compactness, labels, centers = cv2.kmeans(
            data,
            2,
            None,
            criteria,
            5,
            cv2.KMEANS_PP_CENTERS
        )

        # İlk fit'ten sonra label sırası değişmesin diye yeni merkezleri eski merkezlere eşle.
        if self.centers is not None:
            old = self.centers
            d00 = np.linalg.norm(centers[0] - old[0]) + np.linalg.norm(centers[1] - old[1])
            d01 = np.linalg.norm(centers[0] - old[1]) + np.linalg.norm(centers[1] - old[0])
            if d01 < d00:
                centers = centers[[1, 0]]

        self.centers = centers.astype(np.float32)
        self.new_samples_since_fit = 0

    def add_feature(self, feature):
        if feature is None:
            return
        self.features.append(np.asarray(feature, dtype=np.float32))
        self.new_samples_since_fit += 1
        if self.centers is None or self.new_samples_since_fit >= self.refit_every:
            self._fit()

    def predict_team_id(self, feature):
        if feature is None or self.centers is None:
            return None
        feature = np.asarray(feature, dtype=np.float32)
        distances = np.linalg.norm(self.centers - feature[None, :], axis=1)
        return int(np.argmin(distances)) + 1  # 1 veya 2

    def update_track(self, track_id, feature):
        self.add_feature(feature)
        team_id = self.predict_team_id(feature)

        if track_id is None:
            return team_id

        track_id = int(track_id)
        if team_id is not None:
            self.track_votes[track_id][team_id] += 1

        total_votes = sum(self.track_votes[track_id].values())
        if total_votes >= self.min_track_votes:
            self.track_id_to_team[track_id] = self.track_votes[track_id].most_common(1)[0][0]

        return self.track_id_to_team.get(track_id, team_id)

    def get_track_team(self, track_id):
        if track_id is None:
            return None
        return self.track_id_to_team.get(int(track_id))


def team_label(team_id):
    if team_id == 1:
        return TEAM_1_NAME
    if team_id == 2:
        return TEAM_2_NAME
    return UNKNOWN_TEAM_NAME


def should_team_classify(class_name):
    if class_name == "player":
        return True
    if class_name == "goalkeeper" and CLASSIFY_GOALKEEPERS:
        return True
    return False


def update_team_assignments(frame, tracked_detections, team_classifier):
    """
    Her detection için takım bilgisini günceller.
    Dönüş: index -> team_id sözlüğü.
    """
    index_to_team = {}
    if not TEAM_CLASSIFICATION_ENABLED or team_classifier is None:
        return index_to_team
    if tracked_detections is None or len(tracked_detections) == 0:
        return index_to_team

    tracker_ids = getattr(tracked_detections, "tracker_id", None)
    if tracker_ids is None:
        tracker_ids = [None] * len(tracked_detections)

    for i, (xyxy, class_id, track_id) in enumerate(zip(tracked_detections.xyxy, tracked_detections.class_id, tracker_ids)):
        class_name = get_class_name_from_id(class_id)
        if not should_team_classify(class_name):
            continue
        feature = extract_jersey_feature(frame, xyxy)
        team_id = team_classifier.update_track(track_id, feature)
        index_to_team[i] = team_id

    return index_to_team



class StableIDManager:
    """
    ByteTrack raw ID'lerini post-process ederek daha stabil stable_id üretir.

    Mantık:
    - Aynı raw track_id devam ediyorsa stable_id korunur.
    - Raw ID birden değişirse, yakın zamanda kaybolmuş aynı class/takım oyuncusuna yakınsa eski stable_id'ye bağlanır.
    - Raw ID aynı kalsa bile takım rengi veya konum bir anda mantıksız değişirse raw ID "hijack" kabul edilip yeniden eşleştirilir.
    """
    def __init__(
        self,
        fps=25.0,
        max_gap_seconds=1.2,
        player_match_distance_px=135,
        ball_match_distance_px=90,
        referee_match_distance_px=120,
        team_strict=True
    ):
        self.fps = float(fps) if fps else 25.0
        self.max_gap_frames = int(max_gap_seconds * self.fps)
        self.player_match_distance_px = float(player_match_distance_px)
        self.ball_match_distance_px = float(ball_match_distance_px)
        self.referee_match_distance_px = float(referee_match_distance_px)
        self.team_strict = bool(team_strict)

        self.next_stable_id = 1
        self.raw_to_stable = {}
        self.states = {}
        self.updated_in_frame = set()

    def _distance_threshold(self, class_name):
        if class_name == "ball":
            return self.ball_match_distance_px
        if class_name == "referee":
            return self.referee_match_distance_px
        return self.player_match_distance_px

    def _team_compatible(self, state_team, det_team):
        if not self.team_strict:
            return True
        if state_team is None or det_team is None:
            return True
        return int(state_team) == int(det_team)

    @staticmethod
    def _foot_from_xyxy(xyxy):
        x1, y1, x2, y2 = map(float, xyxy)
        return np.array([(x1 + x2) / 2.0, y2], dtype=np.float32)

    def _new_stable(self, raw_id, class_name, team_id, xyxy, frame_idx):
        sid = self.next_stable_id
        self.next_stable_id += 1
        if raw_id is not None:
            self.raw_to_stable[int(raw_id)] = sid

        team_votes = Counter()
        if team_id is not None:
            team_votes[int(team_id)] += 1

        self.states[sid] = {
            "stable_id": sid,
            "raw_ids": set([] if raw_id is None else [int(raw_id)]),
            "class_name": class_name,
            "team_votes": team_votes,
            "team_id": None if team_id is None else int(team_id),
            "last_frame": int(frame_idx),
            "last_xyxy": np.asarray(xyxy, dtype=np.float32),
            "last_foot": self._foot_from_xyxy(xyxy),
        }
        self.updated_in_frame.add(sid)
        return sid

    def _update_state(self, sid, raw_id, class_name, team_id, xyxy, frame_idx):
        st = self.states[sid]
        if raw_id is not None:
            raw_id = int(raw_id)
            self.raw_to_stable[raw_id] = sid
            st["raw_ids"].add(raw_id)

        if team_id is not None:
            st["team_votes"][int(team_id)] += 1
            # Takım etiketi çoğunlukla değişsin ama tek-frame gürültüyle değişmesin.
            st["team_id"] = st["team_votes"].most_common(1)[0][0]

        st["class_name"] = class_name
        st["last_frame"] = int(frame_idx)
        st["last_xyxy"] = np.asarray(xyxy, dtype=np.float32)
        st["last_foot"] = self._foot_from_xyxy(xyxy)
        self.updated_in_frame.add(sid)
        return sid

    def _raw_mapping_valid(self, sid, class_name, team_id, xyxy, frame_idx):
        if sid not in self.states:
            return False
        st = self.states[sid]
        if st["class_name"] != class_name:
            return False
        if not self._team_compatible(st.get("team_id"), team_id):
            return False

        gap = int(frame_idx) - int(st["last_frame"])
        # Aynı frame'de aynı stable_id iki detection'a verilmesin.
        if sid in self.updated_in_frame:
            return False

        # Çok büyük sıçrama varsa raw ID hijack olmuş olabilir.
        dist = float(np.linalg.norm(self._foot_from_xyxy(xyxy) - st["last_foot"]))
        threshold = self._distance_threshold(class_name)
        allowed = threshold * max(1.0, min(3.0, gap + 1))
        return dist <= allowed

    def _find_reconnect_candidate(self, class_name, team_id, xyxy, frame_idx):
        foot = self._foot_from_xyxy(xyxy)
        threshold = self._distance_threshold(class_name)

        best_sid = None
        best_score = None

        for sid, st in self.states.items():
            if sid in self.updated_in_frame:
                continue
            if st["class_name"] != class_name:
                continue

            gap = int(frame_idx) - int(st["last_frame"])
            if gap < 0 or gap > self.max_gap_frames:
                continue

            if not self._team_compatible(st.get("team_id"), team_id):
                continue

            dist = float(np.linalg.norm(foot - st["last_foot"]))
            # Gap arttıkça daha fazla mesafeye izin ver ama yanlış birleşmeyi sınırlı tut.
            allowed = threshold * max(1.0, min(2.5, 1.0 + gap / max(1, self.fps)))
            if dist > allowed:
                continue

            team_bonus = 0.0
            if team_id is not None and st.get("team_id") == int(team_id):
                team_bonus = -25.0

            score = dist + gap * 2.0 + team_bonus
            if best_score is None or score < best_score:
                best_score = score
                best_sid = sid

        return best_sid

    def update(self, xyxy, class_name, raw_track_id, team_id, frame_idx):
        raw_id = None if raw_track_id is None else int(raw_track_id)
        self.updated_in_frame = getattr(self, "updated_in_frame", set())

        if not STABLE_ID_ENABLED:
            return raw_id

        # 1) Önce raw_id'nin mevcut stable eşleşmesi mantıklı mı?
        if raw_id is not None and raw_id in self.raw_to_stable:
            sid = self.raw_to_stable[raw_id]
            if self._raw_mapping_valid(sid, class_name, team_id, xyxy, frame_idx):
                return self._update_state(sid, raw_id, class_name, team_id, xyxy, frame_idx)

        # 2) Raw ID değişmiş olabilir; yakın zamanda kaybolan uygun stable ID'yi bul.
        sid = self._find_reconnect_candidate(class_name, team_id, xyxy, frame_idx)
        if sid is not None:
            return self._update_state(sid, raw_id, class_name, team_id, xyxy, frame_idx)

        # 3) Uygun eşleşme yoksa yeni stable ID aç.
        return self._new_stable(raw_id, class_name, team_id, xyxy, frame_idx)

    def begin_frame(self):
        self.updated_in_frame = set()

    def cleanup(self, frame_idx, keep_seconds=8.0):
        keep_frames = int(keep_seconds * self.fps)
        old_sids = [
            sid for sid, st in self.states.items()
            if int(frame_idx) - int(st["last_frame"]) > keep_frames
        ]
        for sid in old_sids:
            self.states.pop(sid, None)
        # raw_to_stable içinden silinen stable'ları temizle
        valid_sids = set(self.states.keys())
        self.raw_to_stable = {
            raw: sid for raw, sid in self.raw_to_stable.items()
            if sid in valid_sids
        }

def get_color_for_object(class_name, team_id=None):
    """OpenCV BGR renkleri."""
    if class_name == "ball":
        return BALL_COLOR
    if class_name == "referee":
        return REFEREE_COLOR
    if class_name == "goalkeeper":
        # Kaleciyi ayrıca göstermek istersen bu renk kalır.
        if CLASSIFY_GOALKEEPERS and team_id == 1:
            return TEAM_1_DRAW_COLOR
        if CLASSIFY_GOALKEEPERS and team_id == 2:
            return TEAM_2_DRAW_COLOR
        return GOALKEEPER_COLOR
    if class_name == "player":
        if team_id == 1:
            return TEAM_1_DRAW_COLOR
        if team_id == 2:
            return TEAM_2_DRAW_COLOR
        return UNKNOWN_PLAYER_COLOR
    return (255, 255, 255)


# Geriye uyumluluk: tek-frame testte takım bilgisi olmadığı için normal renk döndürür.
def get_color_for_class(class_name):
    return get_color_for_object(class_name, None)


def draw_ellipse_under_object(frame, xyxy, color, thickness=3):
    x1, y1, x2, y2 = map(int, xyxy)
    width = max(10, x2 - x1)
    center_x = int((x1 + x2) / 2)
    bottom_y = int(y2)

    axis_x = max(14, int(width * 0.60))
    axis_y = max(5, int(width * 0.18))

    cv2.ellipse(
        frame,
        center=(center_x, bottom_y),
        axes=(axis_x, axis_y),
        angle=0,
        startAngle=0,
        endAngle=360,
        color=color,
        thickness=thickness,
        lineType=cv2.LINE_AA
    )


def draw_label_box(frame, text, xyxy, color):
    x1, y1, x2, y2 = map(int, xyxy)
    center_x = int((x1 + x2) / 2)
    bottom_y = int(y2)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.55
    thickness = 2

    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    box_w = tw + 12
    box_h = th + 10

    bx1 = int(center_x - box_w / 2)
    by1 = int(bottom_y + 6)
    bx2 = bx1 + box_w
    by2 = by1 + box_h

    h, w = frame.shape[:2]
    bx1 = max(0, min(bx1, w - box_w))
    bx2 = bx1 + box_w
    by1 = max(0, min(by1, h - box_h))
    by2 = by1 + box_h

    cv2.rectangle(frame, (bx1, by1), (bx2, by2), color, -1)
    cv2.putText(
        frame,
        text,
        (bx1 + 6, by2 - 7),
        font,
        font_scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA
    )


def draw_ball_triangle(frame, xyxy, color=(0, 255, 0)):
    x1, y1, x2, y2 = map(int, xyxy)
    center_x = int((x1 + x2) / 2)
    top_y = int(y1)

    pts = np.array([
        [center_x, top_y - 18],
        [center_x - 9, top_y - 36],
        [center_x + 9, top_y - 36]
    ], np.int32)

    cv2.fillPoly(frame, [pts], color)
    cv2.polylines(frame, [pts], True, (0, 0, 0), 2, cv2.LINE_AA)


def draw_detection_view(frame, detections, frame_idx=None, api_time=None):
    """Tracking olmadan tek-frame detection görselleştirme."""
    out = frame.copy()

    for i, d in enumerate(detections):
        xyxy = d["xyxy"]
        class_name = d["class_name"]
        conf = d["confidence"]
        color = get_color_for_class(class_name)

        if class_name == "ball" and DRAW_BALL_TRIANGLE:
            draw_ball_triangle(out, xyxy, color)
        else:
            if DRAW_ELLIPSES:
                draw_ellipse_under_object(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        label_parts = []
        if SHOW_TRACK_ID:
            label_parts.append(str(i + 1))
        if SHOW_CLASS_NAME:
            label_parts.append(class_name)
        if SHOW_CONFIDENCE:
            label_parts.append(f"{conf:.2f}")

        if label_parts:
            draw_label_box(out, " ".join(label_parts), xyxy, color)

    if frame_idx is not None:
        info = f"frame={frame_idx} detections={len(detections)}"
        if api_time is not None:
            info += f" api={api_time:.2f}s"
        cv2.putText(out, info, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 3, cv2.LINE_AA)

    return out


def draw_tracking_view(frame, tracked_detections, index_to_team=None, index_to_display_id=None):
    """Supervision Detections + tracker_id + team_id ile çizim."""
    out = frame.copy()
    if index_to_team is None:
        index_to_team = {}
    if index_to_display_id is None:
        index_to_display_id = {}

    if tracked_detections is None or len(tracked_detections) == 0:
        return out

    xyxys = tracked_detections.xyxy
    class_ids = tracked_detections.class_id
    confidences = tracked_detections.confidence

    tracker_ids = getattr(tracked_detections, "tracker_id", None)
    if tracker_ids is None:
        tracker_ids = [None] * len(xyxys)

    for i, (xyxy, class_id, conf, track_id) in enumerate(zip(xyxys, class_ids, confidences, tracker_ids)):
        class_name = get_class_name_from_id(class_id)
        team_id = index_to_team.get(i, None)
        color = get_color_for_object(class_name, team_id)

        if class_name == "ball":
            if DRAW_BALL_TRIANGLE:
                draw_ball_triangle(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        else:
            if DRAW_ELLIPSES:
                draw_ellipse_under_object(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        label_parts = []
        display_id = index_to_display_id.get(i, track_id)
        if SHOW_TRACK_ID and display_id is not None:
            label_parts.append(str(int(display_id)))
        if SHOW_TEAM_LABEL and team_id is not None and class_name == "player":
            label_parts.append("T" + str(int(team_id)))
        if SHOW_CLASS_NAME:
            label_parts.append(class_name)
        if SHOW_CONFIDENCE and conf is not None:
            label_parts.append(f"{float(conf):.2f}")

        if label_parts:
            draw_label_box(out, " ".join(label_parts), xyxy, color)

    return out


## 5) Tek frame testi

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Video açılamadı.")

source_fps = cap.get(cv2.CAP_PROP_FPS)
source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / source_fps if source_fps and source_fps > 0 else None

print("FPS:", source_fps)
print("Boyut:", source_width, "x", source_height)
print("Toplam frame:", total_frames)
print("Süre:", duration)

ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("İlk frame okunamadı.")

test_frame_path = str(OUTPUT_DIR / "test_frame.jpg")
cv2.imwrite(test_frame_path, frame)

t0 = time.time()
result = CLIENT.infer(test_frame_path, model_id=MODEL_ID)
api_time = time.time() - t0

detections = roboflow_predictions_to_xyxy(result, CONFIDENCE_THRESHOLD)
annotated = draw_detection_view(frame, detections, frame_idx=0, api_time=api_time)

test_output_path = str(OUTPUT_DIR / "test_frame_annotated.jpg")
cv2.imwrite(test_output_path, annotated)

print("Detection sayısı:", len(detections))
print("API süresi:", api_time)

display(HTML(f'<img src="{test_output_path}" width="100%">'))

## 6) Smooth video oluşturma + takım etiketli CSV + stabil ID

Bu hücre videoyu işler, takım sınıflandırıcısını online olarak günceller ve CSV'ye `team_id` / `team_label` kolonlarını yazar.

Bu sürümde ayrıca `StableIDManager` var. ByteTrack'in raw ID'si çakışma/occlusion sırasında değişirse, oyuncunun son konumu ve takım rengi kullanılarak eski `stable_track_id` ile yeniden bağlanmaya çalışılır.

CSV'de:
- `track_id`: geriye uyumluluk için artık `stable_track_id` ile aynı yazılır.
- `stable_track_id`: istatistik çıkarırken kullanman gereken ID.
- `raw_track_id`: ByteTrack'in ham ID'si. Debug için tutulur.

İlk birkaç frame'de bazı oyuncular `unknown` görünebilir. `TEAM_MIN_SAMPLES` kadar oyuncu örneği toplandıktan sonra takım renkleri oturur.

In [ ]:
import supervision as sv

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Video açılamadı.")

source_fps = cap.get(cv2.CAP_PROP_FPS)
if source_fps is None or source_fps <= 0 or math.isnan(source_fps):
    source_fps = 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if MAX_SECONDS is None:
    max_frames = total_frames
else:
    max_frames = min(total_frames, int(MAX_SECONDS * source_fps))

print(f"Kaynak FPS: {source_fps}")
print(f"İşlenecek frame: {max_frames}")
print(f"Detection aralığı: Her {DETECT_EVERY_N_FRAMES} frame")
print(f"Takım ayrımı: {'Açık' if TEAM_CLASSIFICATION_ENABLED else 'Kapalı'}")

output_video_path = str(OUTPUT_DIR / "football_tracking_team_colored.mp4")
output_csv_path = str(OUTPUT_DIR / "football_tracking_team_results.csv")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_video_path, fourcc, source_fps, (width, height))

tracker = sv.ByteTrack(frame_rate=int(round(source_fps)))
team_classifier = OnlineTeamClassifier(
    min_samples=TEAM_MIN_SAMPLES,
    max_samples=TEAM_MAX_SAMPLES,
    refit_every=TEAM_REFIT_EVERY,
    min_track_votes=TEAM_MIN_TRACK_VOTES
)

stable_id_manager = StableIDManager(
    fps=source_fps,
    max_gap_seconds=STABLE_MAX_GAP_SECONDS,
    player_match_distance_px=STABLE_PLAYER_MATCH_DISTANCE_PX,
    ball_match_distance_px=STABLE_BALL_MATCH_DISTANCE_PX,
    referee_match_distance_px=STABLE_REFEREE_MATCH_DISTANCE_PX,
    team_strict=STABLE_TEAM_STRICT
)

last_tracked = sv.Detections.empty()
last_index_to_team = {}
last_index_to_stable_id = {}
records = []

temp_frame_path = str(OUTPUT_DIR / "_temp_frame.jpg")

for frame_idx in tqdm(range(max_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    stable_id_manager.begin_frame()
    should_detect = (frame_idx % DETECT_EVERY_N_FRAMES == 0)
    api_time = None

    if should_detect:
        cv2.imwrite(temp_frame_path, frame)

        t0 = time.time()
        result = CLIENT.infer(temp_frame_path, model_id=MODEL_ID)
        api_time = time.time() - t0

        raw_detections = roboflow_predictions_to_xyxy(result, CONFIDENCE_THRESHOLD)
        sv_detections = detections_to_supervision(raw_detections)

        # ByteTrack güncelle
        last_tracked = tracker.update_with_detections(sv_detections)

        # Takım atamalarını güncelle
        last_index_to_team = update_team_assignments(frame, last_tracked, team_classifier)

        # Stable ID atamalarını güncelle. Videoda ve CSV'de bu ID'leri kullanmak ID switch etkisini azaltır.
        last_index_to_stable_id = {}
        tracker_ids_for_stable = getattr(last_tracked, "tracker_id", None)
        if tracker_ids_for_stable is None:
            tracker_ids_for_stable = [None] * len(last_tracked)

        for stable_i, (stable_xyxy, stable_class_id, stable_raw_id) in enumerate(zip(
            last_tracked.xyxy,
            last_tracked.class_id,
            tracker_ids_for_stable
        )):
            stable_class_name = get_class_name_from_id(stable_class_id)
            stable_team_id = last_index_to_team.get(stable_i, None)
            stable_id = stable_id_manager.update(
                xyxy=stable_xyxy,
                class_name=stable_class_name,
                raw_track_id=stable_raw_id,
                team_id=stable_team_id,
                frame_idx=frame_idx
            )
            last_index_to_stable_id[stable_i] = stable_id

        stable_id_manager.cleanup(frame_idx)

        # CSV kayıtları
        tracker_ids = getattr(last_tracked, "tracker_id", None)
        if tracker_ids is None:
            tracker_ids = [None] * len(last_tracked)

        for i, (xyxy, conf, class_id, track_id) in enumerate(zip(
            last_tracked.xyxy,
            last_tracked.confidence,
            last_tracked.class_id,
            tracker_ids
        )):
            x1, y1, x2, y2 = map(float, xyxy)
            class_name = get_class_name_from_id(class_id)
            team_id_value = last_index_to_team.get(i, None)

            # Oyuncunun sahadaki pozisyonu için ayak noktası, center'dan daha anlamlıdır.
            foot_x = (x1 + x2) / 2
            foot_y = y2

            stable_id_value = last_index_to_stable_id.get(i, None)

            records.append({
                "frame": frame_idx,
                "time_sec": frame_idx / source_fps,
                # Geriye uyumluluk için track_id artık stable_id olarak yazılıyor.
                "track_id": None if stable_id_value is None else int(stable_id_value),
                "stable_track_id": None if stable_id_value is None else int(stable_id_value),
                "raw_track_id": None if track_id is None else int(track_id),
                "class_id": int(class_id),
                "class_name": class_name,
                "team_id": team_id_value,
                "team_label": team_label(team_id_value) if should_team_classify(class_name) else "",
                "confidence": None if conf is None else float(conf),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": (x1 + x2) / 2,
                "center_y": (y1 + y2) / 2,
                "foot_x": foot_x,
                "foot_y": foot_y,
                "api_time": api_time,
                "team_model_ready": bool(team_classifier.ready)
            })

    annotated = draw_tracking_view(frame, last_tracked, last_index_to_team, last_index_to_stable_id if SHOW_STABLE_ID else None)

    # İstersen durum yazısı açabilirsin.
    # status = f"frame={frame_idx} team_model={'ready' if team_classifier.ready else 'warmup'}"
    # cv2.putText(annotated, status, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)

    writer.write(annotated)

cap.release()
writer.release()

if os.path.exists(temp_frame_path):
    os.remove(temp_frame_path)

df = pd.DataFrame(records)
df.to_csv(output_csv_path, index=False)

print("Video kaydedildi:", output_video_path)
print("CSV kaydedildi:", output_csv_path)
print("Kayıt sayısı:", len(df))
print("Takım modeli hazır mı:", team_classifier.ready)
print("Team ID dağılımı:")
if len(df) > 0 and "team_label" in df.columns:
    display(df["team_label"].value_counts(dropna=False).to_frame("count"))


## 7) Colab içinde çıktı videosunu oynat

In [ ]:
import os
import subprocess
from IPython.display import Video, display

def make_colab_playable_mp4(input_path):
    input_path = str(input_path)
    playable_path = input_path.replace(".mp4", "_playable.mp4")

    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        playable_path,
    ]

    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("ffmpeg dönüştürme başarısız oldu.")

    return playable_path

playable_video_path = make_colab_playable_mp4(output_video_path)

print("Oynatılabilir video:", playable_video_path)
print("Dosya boyutu MB:", round(os.path.getsize(playable_video_path) / (1024 * 1024), 2))

display(Video(playable_video_path, embed=True, width=960))


## 8) Çıktıları indir

In [ ]:
files.download(playable_video_path)
files.download(output_csv_path)

## Sorun giderme

### Bazı oyuncular kaçıyorsa
- `DETECT_EVERY_N_FRAMES = 1` kullan.
- `CONFIDENCE_THRESHOLD = 0.15` veya `0.20` dene.
- En iyi sonuç için kendi maç görüntülerinden Roboflow dataset'i ile modeli fine-tune et.

### Takım renkleri karışıyorsa
- `TEAM_MIN_SAMPLES` değerini 40-60 yap.
- `SHOW_TEAM_LABEL = True` açıp hangi oyuncunun T1/T2 aldığını kontrol et.
- Forma bölgesi iyi yakalanmıyorsa `extract_jersey_feature()` içindeki crop oranlarını değiştir.
- Hakem veya kaleci karışıyorsa `CLASSIFY_GOALKEEPERS = False` kalsın.

### Team 1 ve Team 2 renkleri ters geldiyse
Bu normaldir; KMeans kümeleri otomatik oluşturur. Sadece şu renkleri değiştirmen yeterli:

```python
TEAM_1_DRAW_COLOR = (0, 0, 255)
TEAM_2_DRAW_COLOR = (255, 0, 0)
```

### ID'ler çok değişiyorsa
- `DETECT_EVERY_N_FRAMES = 1` yap.
- Daha uzun analizlerde Roboflow API yerine yerel YOLO + BoT-SORT / StrongSORT daha stabil olur.

### İstatistik için sonraki adım
CSV'deki `foot_x`, `foot_y` piksel koordinatıdır. Gerçek mesafe, hız ve heatmap için homography ile saha metre koordinatına çevirmek gerekir.


### ID 4. saniyede 7 iken 5. saniyede 19 gibi değişiyorsa
Bu bir **ID switch** problemidir. Bu notebook'ta `STABLE_ID_ENABLED = True` ile post-process stabil ID katmanı eklendi.

Ayarlar:
- `STABLE_PLAYER_MATCH_DISTANCE_PX`: Aynı oyuncuyu tekrar bağlamak için piksel mesafesi. Çok fazla ID bölünüyorsa artır; yanlış oyuncuları birleştiriyorsa azalt.
- `STABLE_MAX_GAP_SECONDS`: Oyuncu kaç saniyeye kadar kaybolup geri gelirse aynı kişi kabul edilsin.
- `STABLE_TEAM_STRICT = True`: Farklı takım renkleri eşleşmesin diye açık kalması önerilir.

İstatistik çıkarırken mutlaka `stable_track_id` kullan. `raw_track_id` sadece ByteTrack'in ham çıktısıdır.